# Unified Zeek TSV Preprocessing Notebook

This notebook standardizes preprocessing for Zeek-generated TSV files.

In [1]:
import sys

sys.path.insert(0, '..')

from __future__ import annotations
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from utils.utils import *

In [2]:
def normalize_label_name(label: str) -> str:
    label = str(label).strip()
    return (
        label.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .upper()
    )

def clean_numeric_columns(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    cols = [col for col in numeric_cols if col in df.columns]

    df[cols] = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )
    return df

def compute_time_elapsed(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["id.orig_h", "id.resp_h", "ts"]).reset_index(drop=True)
    df["time_elapsed"] = df.groupby(["id.orig_h", "id.resp_h"])["ts"].diff().fillna(999999.0)
    return df

def compute_valid_tcp_handshake(df: pd.DataFrame) -> pd.DataFrame:
    proto_ok = df["proto"].eq("tcp")
    history_ok = df["history"].str.contains(
        r"(?=.*S)(?=.*h)(?=.*A)",
        regex=True,
        na=False
    )
    df["valid_tcp_handshake"] = (proto_ok & history_ok).astype(int)
    return df


def compute_valid_http_conn(df: pd.DataFrame) -> pd.DataFrame:
    service = df["service"].astype(str).str.lower()

    proto_ok = df["proto"].eq("tcp")
    http_ok = df["id.resp_p"].isin([80, 8080, 8000]) & service.eq("http")
    https_ok = df["id.resp_p"].isin([443, 8443]) & service.isin(["https", "ssl"])
    has_data = df["history"].str.contains(r"D", regex=True, na=False)

    df["valid_http_conn"] = (proto_ok & (http_ok | https_ok) & has_data).astype(int)
    return df

def compute_portscan_window_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    failed_states = {"S0", "REJ", "RSTO", "RSTR", "RSTOS0", "RSTRH", "SH", "SHR"}

    df["window_id"] = (df["ts"] // window_seconds).astype(int)
    df["is_failed_conn"] = df["conn_state"].astype(str).isin(failed_states).astype(int)

    agg = (
        df.groupby(["id.orig_h", "window_id"])
        .agg(
            uniq_dst_ports=("id.resp_p", "nunique"),
            total_orig_pkts=("orig_pkts", "sum"),
            scan_duration=("duration", "max"),
            total_flows=("id.orig_h", "size"),
            failed_flows=("is_failed_conn", "sum"),
        )
        .reset_index()
    )

    agg["pkts_per_port"] = (agg["total_orig_pkts"] / agg["uniq_dst_ports"].replace(0, np.nan)).fillna(0.0)
    agg["fail_ratio"] = (agg["failed_flows"] / agg["total_flows"].replace(0, np.nan)).fillna(0.0)

    return df.merge(
        agg[["id.orig_h", "window_id", "uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]],
        on=["id.orig_h", "window_id"],
        how="left",
    )


def add_engineered_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    print_section("Adding engineered features")
    print(f"input rows: {len(df):,}")

    df = clean_numeric_columns(df, MODEL_NUMERIC_FEATURES)
    df = compute_time_elapsed(df)
    df = compute_valid_tcp_handshake(df)
    df = compute_valid_http_conn(df)
    df = compute_portscan_window_features(df, window_seconds=window_seconds)
    df = clean_numeric_columns(df, ENGINEERED_FEATURES)

    print(f"output rows: {len(df):,}")
    print("engineered features:", ", ".join(ENGINEERED_FEATURES))
    return df

def drop_invalid_http_flood_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    is_dos = df["label"] == "DOS_HTTP_FLOOD"
    not_tcp = df["proto"].astype(str).str.lower().ne("tcp")
    not_correct_service = ~df["service"].astype(str).str.lower().isin({"http", "https", "ssl"})

    remove_mask = is_dos & ( not_tcp |  not_correct_service)

    removed = remove_mask.sum()
    print(f"Removed {removed} DOS_HTTP_FLOOD rows")

    return df.loc[~remove_mask].reset_index(drop=True)

In [3]:
def load_and_prepare_dataset(
    datapath: str,
    target_labels: list[str],
    window_seconds: float = 5.0,
) -> pd.DataFrame:
    print(f"Loading data from: {datapath}")
    df = pd.read_csv(datapath, delimiter="\t", on_bad_lines="skip")
    df.columns = df.columns.str.strip()

    print("Raw shape:", df.shape)

    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df["label"] = df["label"].astype(str).map(normalize_label_name)
    target_labels = [normalize_label_name(x) for x in target_labels]
    df = df[df["label"].isin(target_labels)].copy()

    df = drop_invalid_http_flood_rows(df)
    df = add_engineered_features(df, window_seconds=window_seconds)


    print("Processed shape:", df.shape)
    if "label" in df.columns:
        print(df["label"].value_counts())

    return df

In [14]:
WINDOW_SECONDS = 5.0
TARGET_LABELS = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN", "XSS", "SQL_INJECTION", "BRUTE_FORCE"]

def load_cicids2017_data() -> pd.DataFrame:
    print("Loading CICIDS2017 data...")
    df_cicids2017_tuesday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/tuesday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicids2017_wednesday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/wednesday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicids2017_thursday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/thursday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicids2017_friday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/friday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    res = pd.concat([df_cicids2017_tuesday, df_cicids2017_wednesday, df_cicids2017_thursday, df_cicids2017_friday], ignore_index=True)
    print("Processed shape:", res.shape)
    print(res["label"].value_counts())
    return res

def load_ciciot2023_data() -> pd.DataFrame:
    print("Loading CICIoT2023 data...")
    res = load_and_prepare_dataset(
        datapath="../data/CICIoT2023/ciciot2023_labeled_conn.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    print("Processed shape:", res.shape)
    if "label" in res.columns:
        print(res["label"].value_counts())
    return res

In [15]:
def downsample_label(df: pd.DataFrame, label: str, frac: float) -> pd.DataFrame:
    label_df = df[df["label"] == label]
    df_other = df[df["label"] != label]
    label_df = label_df.sample(frac=frac, random_state=42)
    combined_df = pd.concat([label_df, df_other])
    return combined_df.reset_index(drop=True)

In [16]:
df_cicids2017 = load_cicids2017_data()

Loading CICIDS2017 data...
Loading data from: ../data/CICIDS2017/tuesday_labeled.tsv
Raw shape: (323342, 23)
Removed 0 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 323,342
output rows: 323,342
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (323342, 32)
label
BENIGN         316371
BRUTE_FORCE      6971
Name: count, dtype: int64
Loading data from: ../data/CICIDS2017/wednesday_labeled.tsv
Raw shape: (509362, 23)
Removed 8102 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 473,710
output rows: 473,710
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (473710, 32)
label
BENIGN            318361
DOS_HTTP_FLOOD    155349
Name: count, dtype: int64
Loading data from: ../data/CICIDS2017/thursday_labeled.tsv
Raw sha

In [17]:
df_ciciot2023 = load_ciciot2023_data()

Loading CICIoT2023 data...
Loading data from: ../data/CICIoT2023/ciciot2023_labeled_conn.tsv


C:\Users\Rasmus\AppData\Local\Temp\ipykernel_15508\4199690798.py:7: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(datapath, delimiter="\t", on_bad_lines="skip")


Raw shape: (9951064, 23)
Removed 1363138 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 720,842
output rows: 720,842
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (720842, 32)
label
BENIGN            342255
PORTSCAN          216533
DOS_HTTP_FLOOD    145451
BRUTE_FORCE         7658
SQL_INJECTION       5598
XSS                 3347
Name: count, dtype: int64
Processed shape: (720842, 32)
label
BENIGN            342255
PORTSCAN          216533
DOS_HTTP_FLOOD    145451
BRUTE_FORCE         7658
SQL_INJECTION       5598
XSS                 3347
Name: count, dtype: int64


In [18]:
df_ciciot2023.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,label,time_elapsed,valid_tcp_handshake,valid_http_conn,window_id,is_failed_conn,uniq_dst_ports,pkts_per_port,scan_duration,fail_ratio
0,1.664210e+09,Cl9c2N3D5Vc9mY8u7l,0.0.0.0,68,255.255.255.255,67,udp,dhcp,55.394996,3912.0,...,PORTSCAN,999999.000000,0,0,332841972,1,1,13.0,55.394996,1.0
1,1.664210e+09,CGBvuv1Zwk1oCHWCa,0.0.0.0,68,255.255.255.255,67,udp,dhcp,103.427642,6324.0,...,PORTSCAN,305.609078,0,0,332842033,1,1,21.0,103.427642,1.0
2,1.664211e+09,CiIWzD4JV5RKGVSlL9,0.0.0.0,68,255.255.255.255,67,udp,dhcp,99.483682,6626.0,...,PORTSCAN,353.583207,0,0,332842104,1,1,22.0,99.483682,1.0
3,1.664211e+09,C6pWuK1IhobR9JbrDf,0.0.0.0,68,255.255.255.255,67,udp,dhcp,104.475831,7234.0,...,PORTSCAN,434.849972,0,0,332842191,1,1,24.0,104.475831,1.0
4,1.664211e+09,CZSKnlXNx9dZLdYpi,0.0.0.0,68,255.255.255.255,67,udp,dhcp,88.572202,6328.0,...,PORTSCAN,288.525005,0,0,332842248,1,1,21.0,88.572202,1.0


In [20]:
df_cicids2017.to_csv(f"cicids2017_preprocessed.tsv", sep='\t', index=False, header=True)

In [ ]:
df_ciciot2023.to_csv(f"ciciot2023_preprocessed.tsv", sep='\t', index=False, header=True)

# Create small datasets

In [23]:
BIG_LABEL = ["BENIGN", "PORTSCAN", "DOS_HTTP_FLOOD"]

def downsample_labels(df, TARGET_LABELS, frac=0.1):
    for label in TARGET_LABELS:
        df = downsample_label(df, label=label, frac=frac)
    print(df["label"].value_counts())
    return df.reset_index(drop=True)

cicids_small = downsample_labels(df_cicids2017, BIG_LABEL, frac=0.1)
cicids_small.to_csv(f"cicids2017_preprocessed_small.tsv", sep="\t", index=False, header=True)
ciciot_small = downsample_labels(df_ciciot2023, BIG_LABEL, frac=0.1)
ciciot_small.to_csv(f"ciciot2023_preprocessed_small.tsv", sep="\t", index=False, header=True)

label
BENIGN            128618
PORTSCAN           16122
DOS_HTTP_FLOOD     15535
BRUTE_FORCE         8338
XSS                  672
SQL_INJECTION         18
Name: count, dtype: int64
label
BENIGN            34226
PORTSCAN          21653
DOS_HTTP_FLOOD    14545
BRUTE_FORCE        7658
SQL_INJECTION      5598
XSS                3347
Name: count, dtype: int64


In [ ]:
cicids_small.head()